# 01 — Nettoyage des identités étudiantes

Cette étape ajoute un `stage_id` déterministe, applique les corrections manuelles documentées, parse `nom`/`prenom`, re-segmente les prénoms concaténés et canonise uniquement les variations sûres. Les groupes `divergence_reelle` restent inchangés et sont marqués `needs_review`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.config import load_csv_config
from stages_cleaning.identity import apply_identity_cleaning
from stages_cleaning.provenance import build_stage_id
from stages_cleaning.paths import dataset_paths

DATASET = dataset_paths(ROOT)
print("Jeu de données :", DATASET.name)
RAW_PATH = DATASET.raw
OUTPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_name.csv"
OVERRIDE_PATH = DATASET.identity_overrides
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW_PATH, sep=";", dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
raw.insert(0, "stage_id", build_stage_id(raw))
overrides = load_csv_config(OVERRIDE_PATH, {"source_identity", "corrected_identity", "reason"})
print("Shape brute :", raw.shape)
print("Overrides identité :", len(overrides))

## 1. Nettoyage et traçabilité

In [ ]:
df = apply_identity_cleaning(raw, overrides)
print(df["identity_variant_category"].value_counts(dropna=False))
print(df["identity_final_status"].value_counts(dropna=False))
print("Corrections manuelles :", int(df["identity_manual_corrected"].sum()))
print("Re-segmentations email :", int(df["identity_email_resegmented"].sum()))
print("Canonisations sûres :", int(df["identity_canonicalized"].sum()))

## 2. Vérifications de sécurité

In [ ]:
unsafe = df["identity_variant_category"].eq("divergence_reelle")
assert not df.loc[unsafe, "identity_canonicalized"].any(), "Une divergence réelle a été canonisée"
assert df.loc[unsafe, "identity_requires_review"].all(), "Une divergence réelle n'est pas marquée needs_review"
print("Divergences réelles conservées :", int(unsafe.sum()), "lignes")
display(df.loc[unsafe, ["stage_id", "etudiant", "email", "nom", "prenom", "identity_final_status"]].sort_values("email"))

## 3. Dédoublonnage exact et export

In [ ]:
source_cols = list(raw.columns)
n_before = len(df)
df = df.drop_duplicates(subset=source_cols, keep="first").reset_index(drop=True)
assert df["stage_id"].is_unique, "stage_id doit être unique après suppression des doublons exacts"
front = ["stage_id", "prenom", "nom"]
df = df[front + [c for c in df.columns if c not in front]]
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Doublons exacts supprimés :", n_before - len(df))
print("Export :", OUTPUT_PATH)
print("Shape :", df.shape)